# GTEx QTL Colocalization Prep
This notebook can be used to prepare tissue-level GTEx QTL summary statistics for colocalization. The the GP2 Subtypes and Mechanisms colocalization workflow handles standardization of summary statistics, including liftover (if necessary), dbSNP validation, calculating missing summary statistics, and standardizing format and column names. This notebook can be used to pre-process GTEx QTL parquet/eGenes files so that summary stats are in the expected input format for the colocalization workflow. It will also produce and GTEx QTL manifest file that can be used to drive the colocalization workflow for all of GTEx at once, looping through all tissues and processing independently.

In [12]:
# Imports
import pandas as pd
from pathlib import Path

## Configuration

In [13]:
# Configuration
tissue_sample_counts_fp = '/mnt/working/locus_reports/qtl/eqtl/gtex/significant/tissue_sample_counts.tsv'
gtex_dir = '/mnt/working/locus_reports/qtl/eqtl/gtex/significant/GTEx_Analysis_v10_eQTL_updated'
gtex_tissue_manifest_fn = 'GTEx_tissue_manifest.tsv'

## Load Inputs

In [14]:
# Load tissue sample counts
tissue_sample_counts = pd.read_table(tissue_sample_counts_fp)

## Define Main Function

In [15]:
def process_tissue(tissue):
    """
    Process a single tissue's eQTL data.
    
    Parameters:
    tissue (str): The name of the tissue to process.
    
    Returns:
    pd.DataFrame: A DataFrame containing the processed eQTL data for the tissue.
    """
    print(f"Processing tissue: {tissue}")

    # Define File Paths
    egenes_fp = Path(f"{gtex_dir}/{tissue}.v10.eGenes.txt.gz")
    signif_pairs_fp = Path(f"{gtex_dir}/{tissue}.v10.eQTLs.signif_pairs.parquet")

    if egenes_fp.exists() and signif_pairs_fp.exists():
        
        # Load Data
        egenes = pd.read_table(egenes_fp, compression='gzip')
        signif_pairs = pd.read_parquet(signif_pairs_fp)

        # Merge DataFrames on 'variant_id'
        eqtl_data = signif_pairs.merge(egenes[['variant_id', 'gene_name', 'chr', 'variant_pos', 'ref', 'alt', 'rs_id_dbSNP155_GRCh38p13']], on='variant_id', how='left')

        # Add tissue information
        eqtl_data['tissue'] = tissue

        print(f"Finished processing tissue: {tissue}")
    
        return eqtl_data
    
    else:
        print(f"Files for tissue {tissue} do not exist.")
        return pd.DataFrame()

## Process Tissues

In [16]:
# Process each tissue and save the processed data
for tissue in tissue_sample_counts['Tissue']:
    
    # Process Tissue
    eqtl_data = process_tissue(tissue)

    if not eqtl_data.empty:

        # Define Output File Path
        output_fp = Path(f"{gtex_dir}/{tissue}_cis_qtls_prepared.tsv")

        # Add file path to DataFrame
        tissue_sample_counts.loc[tissue_sample_counts['Tissue'] == tissue, 'file_path'] = str(output_fp)

        # Save Processed Data
        eqtl_data.to_csv(output_fp, sep='\t', index=False)

# Drop rows with missing file paths
tissue_sample_counts = tissue_sample_counts.dropna(subset=['file_path'])

# Save the updated tissue sample counts with file paths
tissue_sample_counts.to_csv(Path(gtex_dir, gtex_tissue_manifest_fn), sep='\t', index=False)
    

Processing tissue: Adipose_Subcutaneous
Finished processing tissue: Adipose_Subcutaneous
Processing tissue: Adipose_Visceral_Omentum
Finished processing tissue: Adipose_Visceral_Omentum
Processing tissue: Adrenal_Gland
Finished processing tissue: Adrenal_Gland
Processing tissue: Artery_Aorta
Finished processing tissue: Artery_Aorta
Processing tissue: Artery_Coronary
Finished processing tissue: Artery_Coronary
Processing tissue: Artery_Pulmonary
Files for tissue Artery_Pulmonary do not exist.
Processing tissue: Artery_Tibial
Finished processing tissue: Artery_Tibial
Processing tissue: Bladder
Finished processing tissue: Bladder
Processing tissue: Brain_Amygdala
Finished processing tissue: Brain_Amygdala
Processing tissue: Brain_Anterior_cingulate_cortex_BA24
Finished processing tissue: Brain_Anterior_cingulate_cortex_BA24
Processing tissue: Brain_Caudate_basal_ganglia
Finished processing tissue: Brain_Caudate_basal_ganglia
Processing tissue: Brain_Cerebellar_Hemisphere
Finished processi